
# External data

**More is more?**
Nearly every top-20 finisher in the real challenge added external public data,
and four of the top five also had proprietary data!

But external data can be challenging to integrate:
a number from someone else's assay is not the same measurement as a number
from yours, and making them comparable can be fiddly.

In [ ]:
#@title Getting things all setup...
# Run me first.
%pip -q install rdkit pandas numpy scipy scikit-learn huggingface_hub fsspec lightgbm matplotlib seaborn pyarrow
!git clone https://github.com/agura-alt/ai4chem_openadmet.git
%cd ai4chem_openadmet


In [ ]:
#@title Imports...
import os, sys
SETUP_DIR = os.path.abspath("Setup")
os.path.isdir(SETUP_DIR) or sys.exit(f"No Setup dir at {SETUP_DIR}; cwd is {os.getcwd()}")

if SETUP_DIR not in sys.path:
    sys.path.insert(0, SETUP_DIR)

assert os.path.exists("Setup/common.py") and os.path.getsize("Setup/common.py") > 1000, (
    "common.py is missing or truncated. Upload it using the folder icon in the "
    "left sidebar, then re-run this cell.")

sys.modules.pop("common", None)            # force a fresh read
import common


Change `your-pair-name` to your team name. It has to match the list of registered teams exactly, and be the same in every notebook &mdash; that is what links your work together.

In [ ]:
# Same folder as every other notebook -- splits, predictions, scores.
common.setup(pair="your-pair-name")

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from lightgbm import LGBMRegressor
sns.set_style("whitegrid")

train = common.load_train()
test  = common.load_test()

display(common.list_splits())

In [ ]:
SPLIT = "random"        # <-- change to whichever you like
fold, split_meta = common.load_split(train, name=SPLIT)

train_df = train[(fold == "train").to_numpy()].reset_index(drop=True)
val_df   = train[(fold == "val").to_numpy()].reset_index(drop=True)
print(f"{len(train_df)} train / {len(val_df)} val molecules")

---
## 1. Potential datasets

| source | description | links |
|---|---|---|
| **TDC** (Therapeutics Data Commons) | Caco-2, solubility, PPB, clearance, LogD | [link](https://tdcommons.ai/single_pred_tasks/adme/) |
| **ASAP Discovery / Polaris** antiviral ADMET | LogD, KSOL, HLM, MLM, permeability | [link](https://polarishub.io/datasets/asap-discovery/antiviral-admet-2025-unblinded) |
| **ChEMBL** | enormous, LogD especially | [link](https://www.ebi.ac.uk/chembl/) |
| **Galapagos** released sets | curated public datasets used in the Polaris challenge | [paper](https://chemrxiv.org/doi/10.26434/chemrxiv-2025-q12vh), [zenodo](https://zenodo.org/records/15602150) |
| **BioGen** sets | Published pharmaceutical data | [paper](https://pubs.acs.org/jcisd8/article-abstract/63/11/3263/850292/Prospective-Validation-of-Machine-Learning), [github](https://github.com/molecularinformatics/Computational-ADME) |

This notebook explores incorporating the TDC datasets into our prediction task. Feel free to explore the other external datasets -- but we don't provide any scaffolding for those ones :)

In [ ]:
# The .tab files in Data/artifacts/ were downloaded from Harvard Dataverse
TAB_DIR = os.path.join(common.REPO_DATA_DIR, "artifacts")

SOURCES = {          # our short name  ->  the file, as downloaded from TDC
    "caco2":                "caco2_wang",
    "solubility":           "solubility_aqsoldb",
    "ppb":                  "ppbr_az",
    "clearance_hepatocyte": "clearance_hepatocyte_az",
    "clearance_microsome":  "clearance_microsome_az",
    "logd":                 "lipophilicity_astrazeneca",
}

raw = {name: pd.read_csv(os.path.join(TAB_DIR, stem + ".tab"), sep="\t")
       for name, stem in SOURCES.items()}

for name, df in raw.items():
    print(f"{name:21s} {len(df):6,} rows   columns: {list(df.columns)}")

### &#9654;&#65039; Your turn!

Five files from one source, curated by one group, and they still do not use
the same column names. This is the normal condition of public data, and it's one of the (annoying) costs of integrating multiple data sources.

Read the column lists and inspect the data frames.
Then, fill out the `tidy()` function to align the columns of all five dataframes.

In [ ]:
# let's look at those dfs!
print(raw.keys())
raw["caco2"].head() # switch this out for any other dataframe present!

In [ ]:
def tidy(df):
    """Give those three columns the same name in every file.

    Every other column is left exactly as it was -- including any you have
    not looked at yet.
    """
    ### YOUR CODE ###
    id_cols     = [..., ..., ...]        # names meaning "an identifier for the molecule"
    smiles_cols = [..., ..., ...]        # names meaning "the structure"
    y_cols      = [..., ..., ...]        # names meaning "the measurement"
    ### END YOUR CODE ###

    df = df.rename(columns={name: "source_id" for name in id_cols})
    df = df.rename(columns={name: "SMILES"    for name in smiles_cols})
    df = df.rename(columns={name: "value"     for name in y_cols})
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    return df

ext = {name: tidy(df).assign(source=name) for name, df in raw.items()}

# checking!
for name, df in ext.items():
    assert {"SMILES", "value"} <= set(df.columns), f"{name}: {list(df.columns)}"
    assert pd.api.types.is_numeric_dtype(df["value"]), f"{name}: value is not numeric"
    print(f"{name:11s} {list(df.columns)}")

---
## 2. Map the external sources onto our endpoints

How do we add the external data to our existing dataset? Start with the
simplest question: which of *our* nine endpoints is each external source
trying to measure?


| source | TDC dataset | what it measures |
|---|---|---|
| `logd` | `Lipophilicity_AstraZeneca` | octanol/water distribution at pH 7.4 |
| `solubility` | `Solubility_AqSolDB` | thermodynamic aqueous solubility |
| `caco2` | `Caco2_Wang` | Caco-2 apparent permeability, A&rarr;B |
| `ppb` | `PPBR_AZ` | plasma protein binding |
| `clearance_hepatocyte` | `Clearance_Hepatocyte_AZ` | intrinsic clearance in rat and human **hepatocytes** |
| `clearance_microsome` | `Clearance_Microsome_AZ` | intrinsic clearance in human **microsomes** |

<br><br>
Most rows are a straight read against `common.ENDPOINTS` below, but some aren't!

**Caco-2.** Two of our nine endpoints come off the same Caco-2 plate.

**Clearance.** There are two clearance datasets and one clearance endpoint.
Each dataset is a different experiment:
- **microsomes** are a subcellular fraction, essentially a bag of the
  metabolising enzymes, run per mg of protein
- **hepatocytes** are whole intact cells, run per million cells, so a compound
  has to get through a membrane before anything happens to it

Our clearance endpoints are `Log_HLM_CLint` and `Log_MLM_CLint`, for reference.

In [ ]:
common.ENDPOINTS

In [ ]:
ENDPOINT_MAP = {          # external source -> one of common.ENDPOINTS, or None
    ### YOUR CODE ###
    "logd":                 ...,
    "solubility":           ...,
    "caco2":                ...,
    "ppb":                  ...,
    "clearance_hepatocyte": ...,
    "clearance_microsome":  ...,
    ### END YOUR CODE ###
}

assert set(ENDPOINT_MAP) == set(ext), "one entry per source, no more, no less"
bad = [v for v in ENDPOINT_MAP.values()
       if v is not None and v not in common.ENDPOINTS]
assert not bad, f"not one of common.ENDPOINTS (and not None): {bad}"

for source, endpoint in ENDPOINT_MAP.items():
    print(f"{source:21s} ({len(ext[source]):6,} rows)  ->  {endpoint}")

In [ ]:
#@title log scaling helpers

ALREADY_LOG = {
    "logd": True, "solubility": True, "caco2": True,
    "ppb": False, "clearance_hepatocyte": False, "clearance_microsome": False,
}

# the sources you actually mapped to something
MAPPED = {s: e for s, e in ENDPOINT_MAP.items() if e is not None}

def to_our_scale(df, endpoint):
    """Put external measurements on EXACTLY the scale of our target column.

    Takes a dataframe with `value` and `source` columns -- any frame from
    `ext`, or a cleaned copy of one.

    Matching the units is not enough. `common.load_train` builds every logged
    endpoint as log10((raw + 1) * mult) -- that "+1" is a zero-guard, and it is
    part of the target's definition whether you like it or not. External rows
    that skip it sit on a slightly different curve, and the gap widens exactly
    where the guard bites hardest: our LogS and Log_Caco_Papp_AB are floored at
    -6.0, while AqSolDB reaches -13 and Caco2_Wang -7.7.

    So: undo whatever transform the source used, then redo ours.
    """
    if not len(df):
        return np.array([])
    source = df["source"].iloc[0]
    values = np.asarray(df["value"], dtype=float)

    raw_col = {log_name: raw for raw, (_, _, log_name)
               in common.CONVERSION.items()}[endpoint]
    needs_log, mult, _ = common.CONVERSION[raw_col]
    if ALREADY_LOG[source]:
        if not needs_log:
            return values               # LogD: we never log it, nothing to undo
        raw = 10.0 ** values / mult     # undo our log, WITHOUT the guard
    else:
        raw = values                    # source units are our raw assay units

    if not needs_log:
        return raw
    return np.log10(np.clip(raw + 1.0, 1e-9, None) * mult)   # redo it WITH the guard

---
## 3. What is wrong with that mapping

**There are problems with using the data out of the box.**

While there are certainly more issues than these, here are a few you should try to look for:
- A problem visible in the **distributions**
- A problem visible in the **dataframe**
- A problem with what the assay
  physically *is*

The next few cells are for hunting. Nothing below is an exercise with a right
answer to type in &mdash; they are tools. Use them, change them, add cells.

Start with the distributions, because that is the check you can run without
knowing anything about the assays.

The reliable way to catch a bad mapping is to plot the external distribution
against ours. If the shapes do not line up, something is wrong regardless of
what the documentation claims.

In [ ]:
#@title Plot distributions

fig, axes = plt.subplots(1, len(MAPPED), figsize=(3.6 * len(MAPPED), 3.3))
summary = []
for ax, (source, endpoint) in zip(np.atleast_1d(axes), MAPPED.items()):
    ours = train[endpoint].dropna().to_numpy()
    theirs = to_our_scale(ext[source].dropna(subset=["value"]), endpoint)
    theirs = theirs[np.isfinite(theirs)]

    # shade the span our assay actually reports -- anything outside it is a
    # value our instrument could not have produced
    lo, hi = ours.min(), ours.max()
    ax.axvspan(lo, hi, color="tab:orange", alpha=.10, zorder=0,
               label="range our assay reports")
    bins = np.linspace(min(lo, theirs.min()), max(hi, theirs.max()), 45)
    ax.hist(ours, bins=bins, density=True, color="tab:orange", alpha=.55,
            label=f"ExpansionRx (n={len(ours):,})")
    ax.hist(theirs, bins=bins, density=True, histtype="step", lw=1.8,
            color="tab:blue", label=f"external (n={len(theirs):,})")
    ax.set_title(f"{source}\n-> {endpoint}", fontsize=8)
    ax.legend(fontsize=6, loc="upper left"); ax.set_yticks([])

    iqr = lambda a: np.subtract(*np.percentile(a, [75, 25]))
    _, counts = np.unique(np.round(theirs, 6), return_counts=True)
    # the same structure appearing twice with two answers is a question, not
    # necessarily a fault -- but it is always a question
    dup = ext[source][ext[source]["SMILES"].duplicated(keep=False)]
    spread = (dup.groupby("SMILES")["value"].agg(lambda s: s.max() / s.min())
              if len(dup) else pd.Series(dtype=float))
    summary.append({
        "source": source,
        "median shift": np.median(theirs) - np.median(ours),
        "IQR ratio": iqr(theirs) / iqr(ours) if iqr(ours) else np.nan,
        "% ext outside our range": 100 * np.mean((theirs < lo) | (theirs > hi)),
        "% ext on one value": 100 * counts.max() / len(theirs),
        "% rows repeating a structure": 100 * len(dup) / len(ext[source]),
        "...and disagreeing by": (f"{spread.replace([np.inf], np.nan).median():.1f}x"
                                  if len(spread) else "-"),
    })

plt.tight_layout(); plt.show()
pd.DataFrame(summary).round(2)

Each column of that table detects a different way for a mapping to be wrong:

- **median shift** &mdash; the whole distribution sits somewhere else. Usually
  a units or transform problem, and usually fixable.
- **IQR ratio** &mdash; the spread does not match. Far from 1 means one assay
  has a much wider dynamic range than the other, so they are not
  interchangeable even where they overlap.
- **% ext outside our range** &mdash; external rows holding values our
  instrument never reports. What is a model supposed to learn from a
  measurement your assay could not have produced?
- **% ext on one value** &mdash; how much of the source is piled on a single
  number. A real measurement is rarely repeated to six decimal places.
- **% rows repeating a structure / and disagreeing by** &mdash; the same
  molecule appearing more than once with more than one answer. Sometimes that
  is honest replication. Sometimes it means two different experiments have
  been stacked into one file.

In [ ]:
# You had to pick one clearance dataset before seeing any of this. Here are
# both, against the same endpoint, whichever one you mapped.
fig, axes = plt.subplots(1, 2, figsize=(9, 3.3), sharex=True)
ours = train["Log_HLM_CLint"].dropna().to_numpy()

for ax, source in zip(axes, ["clearance_hepatocyte", "clearance_microsome"]):
    d = ext[source]
    theirs = to_our_scale(d.dropna(subset=["value"]), "Log_HLM_CLint")
    ax.axvspan(ours.min(), ours.max(), color="tab:orange", alpha=.10, zorder=0)
    bins = np.linspace(0, 3.6, 45)
    ax.hist(ours, bins=bins, density=True, color="tab:orange", alpha=.55,
            label="ExpansionRx HLM")
    ax.hist(theirs, bins=bins, density=True, histtype="step", lw=1.8,
            color="tab:blue", label=source)
    ax.set_title(source, fontsize=9); ax.legend(fontsize=7); ax.set_yticks([])
plt.tight_layout(); plt.show()

print(f"{'':22s} {'rows':>7s} {'structures':>11s} {'p50':>7s} {'p75':>7s} {'p90':>7s}")
print(f"{'ExpansionRx HLM CLint':22s} {len(ours):7,} {'':>11s} "
      + "".join(f"{np.percentile(10**ours - 1, q):7.1f}" for q in (50, 75, 90)))
for source in ["clearance_hepatocyte", "clearance_microsome"]:
    v = ext[source]["value"].dropna()
    print(f"{source:22s} {len(v):7,} {ext[source]['SMILES'].nunique():11,} "
          + "".join(f"{np.percentile(v, q):7.1f}" for q in (50, 75, 90)))

In [ ]:
# A scratchpad.
source = "ppb"

display(ext[source].head())
print("columns you have not used:",
      [c for c in ext[source].columns if c not in ("SMILES", "value")])

# for a column that is not a number, what is in it and how much of each?
# for a column that is a number, where does it pile up?
#   ext["clearance"]["value"].value_counts().head()

,source_id,SMILES,value,Species,source
0,CHEMBL1017,CCCc1nc2c(C)cc(-c3nc4ccccc4n3C)cc2n1Cc1ccc(-c2...,98.25,Canis lupus familiaris,ppb
1,CHEMBL2337981,CC(C)(C)NC(=O)NCCN1CCC(CO)(CNC(=O)c2cc(Cl)cc(C...,82.05,Canis lupus familiaris,ppb
2,CHEMBL401158,CCCCNc1nc(SCCC)nc2c1nnn2[C@@H]1C[C@H](CO)[C@@H...,95.33,Canis lupus familiaris,ppb
3,CHEMBL2337980,CC(C)(C)NC(=O)NCCN1CCC(O)(CNC(=O)c2cc(Cl)cc(Cl...,64.01,Canis lupus familiaris,ppb
4,CHEMBL914,CC(C)(C(=O)O)c1ccc(C(O)CCCN2CCC(C(O)(c3ccccc3)...,84.90,Canis lupus familiaris,ppb


columns you have not used: ['source_id', 'Species', 'source']


In [ ]:
#@title Reveal: some problems { display-mode: "form" }
print("--- 1. in the columns -------------------------------------------")
print(ext["ppb"]["Species"].value_counts().to_string())
print("""
PPBR_AZ is five species in one file. Our endpoint is MOUSE plasma protein
binding; a bit over half these rows are human, and the rest are rat, dog and
guinea pig. Nothing in SMILES or value says so -- only the Species column,
which is in front of you because you renamed columns instead of selecting
them. PyTDC's own ADME loader returns Drug and Y and drops the rest, so
anyone who went through it sees 2,828 rows with no hint they are mixed.

That is also why ppb repeats so many structures: the same drug measured in
several species. Those repeats agree closely, because almost everything is
85-99% bound whatever the animal. The Species column turns a confusing
duplicate count into an explanation -- and into a filter.
""")

print("--- 2. in the distributions -------------------------------------")
for s in ["clearance_hepatocyte", "clearance_microsome"]:
    v = ext[s]["value"]
    print(f"  {s:22s} {(v == v.min()).mean():5.1%} at {v.min():>5}, "
          f"{(v == v.max()).mean():5.1%} at {v.max():>5}   "
          f"{ext[s]['SMILES'].nunique():,} structures in {len(v):,} rows")
print("""
Both clearance files censor: values at 3 and at 150 are the assay's limits of
quantitation, reported as if they were measurements. "Below 3" is not a
number, it is the absence of one, and stacking them in teaches a model to
predict the limits of somebody else's instrument. Our own HLM CLint runs to
2,590 with no such pile-up, so the top of our range has no external support.

But look at the structure counts. The hepatocyte file has ~190 compounds
carrying TWO clearance values that differ by a median of 3x -- far too much
for replication. TDC built it from AstraZeneca's ChEMBL deposition, which
contains hepatocyte clearance in more than one species. So it is the same
species problem as ppb with the evidence removed: you can see something is
mixed, but not what, and you cannot filter it. The microsome file has one row
per structure and no such ambiguity.
""")

print("--- 3. differences in what was measured -----------------------------------")
print("""
Every one of these sources measures something ALMOST the same as ours:

  solubility   AqSolDB is THERMODYNAMIC solubility of the neutral compound;
               KSOL is KINETIC solubility from a DMSO stock. Different
               experiment, and it shows: 66% of their values fall outside
               anything our assay reports, at 3x the spread.

  clearance    our endpoint is HLM -- human liver MICROSOMES. One of the two
               files is microsomes and one is hepatocytes, whole cells against
               one subcellular fraction. Does the distribution check separate?

  ppb          PPBR_AZ reports percent BOUND. Our MPPB
               is percent UNBOUND -- median 8.9, max 87.6, which is not what
               "bound" looks like for drug-like compounds. Their mouse rows
               have median 96.2. The two are rank-INVERTED: the most tightly
               bound compound looks like the least bound one. 100 - value
               puts them at median 3.8 against our 8.9, the same regime.

None of that is in the files, and for most of them it is not in TDC's
documentation either. You get it from the dataset NAME, from the upstream
paper, or from knowing the assay.

So: a large distributional difference is not always a mistake (solubility --
the assays genuinely disagree about what they can measure), and the most
consequential facts here left no distributional trace at all. The
distributions, the columns and the names are three different instruments, and
you need all three.
""")

---
## 5. Clean it, then measure

Let's fix some of those problems -- we will work with PPB.

In [ ]:
# Which species are actually in there?
ppb_as_shipped = ext["ppb"] # save the original
print(sorted(ext["ppb"]["Species"].unique()))

['Canis lupus familiaris', 'Cavia porcellus', 'Homo sapiens', 'Mus musculus', 'Rattus norvegicus']


In [ ]:
### YOUR CODE ###
MOUSE = ...              # the one our endpoint measures
### END YOUR CODE ###
ppb_mouse      = ext["ppb"][ext["ppb"]["Species"] == MOUSE] # save mouse filtered

In [ ]:
# save with the bound/unbound issue fixed
ppb_unbound = ppb_as_shipped.copy()
### TODO ###
ppb_unbound["value"] = ...
### END TODO ###

# save with both issues fixed
ppb_unbound_mouse = ppb_mouse.copy()
### TODO ###
ppb_unbound_mouse["value"] = ...
### END TODO ###

In [ ]:
#@title Training helpers...

_DESCRIPTOR_CACHE = {}

def cached_rdkit_descriptors(smiles):
    """common.rdkit_descriptors, but it only computes each molecule once."""
    smiles = list(smiles)
    missing = [s for s in dict.fromkeys(smiles) if s not in _DESCRIPTOR_CACHE]
    if missing:
        X = common.rdkit_descriptors(missing)
        for smi, (_, row) in zip(missing, X.iterrows()):
            _DESCRIPTOR_CACHE[smi] = row
    return pd.DataFrame([_DESCRIPTOR_CACHE[s] for s in smiles]).reset_index(drop=True)


def build_training_set(endpoint, training_data, external=None):
    """ExpansionRx rows for `endpoint`, plus an optional frame of external rows.

    training_data : the frame to train on -- pass train_df while you are
                    experimenting, and the full `train` when you build the
                    model you actually submit. Nothing is read from globals.
    external      : any frame from `ext`, or a cleaned copy of one, or None.
    """
    base = pd.DataFrame({"SMILES": training_data["SMILES"],
                         "y": training_data[endpoint],
                         "origin": "expansionrx"}).dropna()
    if external is None or not len(external):
        return base
    rows = external.dropna(subset=["value"])
    add = pd.DataFrame({"SMILES": rows["SMILES"].to_numpy(),
                        "y": to_our_scale(rows, endpoint),
                        "origin": rows["source"].to_numpy()})
    return pd.concat([base, add], ignore_index=True)


def train_and_score(endpoint, training_data, target_data, external=None):
    """Train one endpoint, return (RAE, rows used). Leaves your score table alone."""
    data = build_training_set(endpoint, training_data, external)
    X_train, X_val = common.clean_features(cached_rdkit_descriptors(data["SMILES"]),
                                           cached_rdkit_descriptors(target_data["SMILES"]))
    model = LGBMRegressor(n_estimators=500, learning_rate=0.05, verbose=-1, n_jobs=-1)
    model.fit(X_train, data["y"])
    preds = pd.DataFrame({"Molecule Name": target_data["Molecule Name"],
                          endpoint: model.predict(X_val)})
    return common.evaluate(target_data, preds, [endpoint]).loc[endpoint, "RAE"], len(data)

In [ ]:
# One endpoint, four training sets. `no external` is the bar the rest must beat.
for label, external in [
    ("no external",    None),
    ("as shipped",     ppb_as_shipped),
    ("mouse only",     ppb_mouse),
    ("mouse, unbound", ppb_unbound_mouse),
    ("unbound only", ppb_unbound)
]:
    rae, n = train_and_score("Log_Mouse_PPB", train_df, val_df, external)
    print(f"  {label:16s} n={n:6,}   RAE = {rae:.3f}")

- Did the models perform better or worse with more data?
- Which did more for the score &mdash; the amount of data or its correctness?
- `mouse only` fixes the species problem and leaves the bound/unbound problem
  in place. Where does it land, and what does that say about fixing one of two
  things?
- Re-run with a different `SPLIT` at the top. Does the ordering hold? Does the
  size of the gap?

### If you have time

- **Settle the clearance question from section 2.** `train_and_score` takes any
  frame as `external=`, so pass `ext["clearance_hepatocyte"]` and
  `ext["clearance_microsome"]` in turn and see whether the choice you reasoned
  your way to shows up in a score.
- Drop the censored rows from whichever clearance file you picked &mdash; the
  ones sitting exactly on the assay's limits &mdash; and see if it matters.
- Weight the external rows down. `LGBMRegressor.fit` takes `sample_weight`; the
  `origin` column in the training set tells you which rows are which.
- Try a different external data source.

---
## 6. Save your work

Everything above scored one endpoint at a time. A submission needs all nine.

Five of the nine have an external source mapped to them; the other four train
on ExpansionRx data alone, because nothing in `ENDPOINT_MAP` points at them.
That is fine &mdash; you still get a prediction for every endpoint.

The cell below applies the one cleaning you measured and takes the other four
sources exactly as shipped. You now know that is not the best you can do. It
is the starting point, and improving it is the rest of your afternoon.

Keep at least one split the same across every notebook today, or the rows in
your score table are not comparable.

In [ ]:
MODEL = "lgbm-external"

# One external frame per endpoint. Everything comes straight from `ext`
# except ppb, where you swap in the version you cleaned and measured.
# if you perform other cleaning, you can swap them in too
EXTERNAL = {endpoint: ext[source] for source, endpoint in MAPPED.items()}
EXTERNAL["Log_Mouse_PPB"] = ppb_unbound_mouse

X_val_raw = cached_rdkit_descriptors(val_df["SMILES"])
preds = pd.DataFrame({"Molecule Name": val_df["Molecule Name"]})
trained = []

for endpoint in common.ENDPOINTS:
    data = build_training_set(endpoint, train_df, EXTERNAL.get(endpoint))
    if len(data) < 50:
        preds[endpoint] = np.nan
        print(f"  {endpoint:20s} skipped -- only {len(data)} labelled rows")
        continue
    X_train, X_val = common.clean_features(cached_rdkit_descriptors(data["SMILES"]), X_val_raw)
    model = LGBMRegressor(n_estimators=500, learning_rate=0.05, verbose=-1, n_jobs=-1)
    model.fit(X_train, data["y"])
    preds[endpoint] = model.predict(X_val)
    trained.append(endpoint)
    n_ext = int((data["origin"] != "expansionrx").sum())
    print(f"  {endpoint:20s} n={len(data):,} ({n_ext:,} external)")

# endpoints=trained so that if anything was skipped, common.score says so
# rather than quietly averaging over a smaller, easier set
metrics = common.score(val_df, preds, MODEL, SPLIT, endpoints=trained,
                       note="LightGBM, external data, ppb cleaned")
metrics.round(3)

In [ ]:
common.score_matrix().round(3)

### Make a submission

Save a submission file. It is
free and unlimited &mdash; nothing counts until you drag a file into the Scored
folder &mdash; and it is also how the `Ensembles` notebook picks this model up
later, so give it a `model` name you will recognise.

You have to name the number you are betting on and the split it came from. If
you do not believe any of your numbers yet, go back and validate differently
rather than guessing.

Go back and try another split if you like, then
make a submission. `MODEL` and `SPLIT` below have to match a row you actually
logged with `common.score` &mdash; `prepare_submission` reads the expected score out of your score
table so you cannot submit a number you never measured.

In [ ]:
# The model you submit trains on EVERY labelled molecule -- pass `train`
# rather than train_df. The split above was only for estimating the score.

MODEL = "lgbm-external"        # <-- must match what you logged above
X_test = cached_rdkit_descriptors(test["SMILES"])

# a submission is just the molecule ids plus one column per endpoint
test_preds = pd.DataFrame({"Molecule Name": test["Molecule Name"]})

for endpoint in common.ENDPOINTS:
    # the training set changes per endpoint: ours, plus whatever
    # EXTERNAL has for it
    data = build_training_set(endpoint, train, EXTERNAL.get(endpoint))
    X_train, X_test = common.clean_features(cached_rdkit_descriptors(data["SMILES"]), X_test)

    model = LGBMRegressor(n_estimators=500, learning_rate=0.05,
                          verbose=-1, n_jobs=-1)
    model.fit(X_train, data["y"])
    test_preds[endpoint] = model.predict(X_test)

common.prepare_submission(test_preds, MODEL, SPLIT,
                          why="LightGBM, external data, ppb cleaned")

---
## Appendix &mdash; is this data anywhere near ours?

Everything above asks whether external measurements mean the same thing as
ours. This asks something different and cheaper: whether the external
*molecules* are close enough to ours to be informative at all. Below a Tanimoto
of about 0.27 a nearest neighbour is indistinguishable from a random molecule.

### Is there anything in this source that looks like my chemistry?

For each of **our** molecules, the cell below finds the most similar molecule in
one external source and reports the median across our set, plus how often the
best match falls below the noise floor. One row per source, so you can rank the
five candidates against each other.

`morgan_fingerprints` returns RDKit fingerprint objects for similarity work &mdash;
not the bit-column DataFrame that `Descriptors.ipynb` builds for modelling.

In [ ]:
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator

def morgan_fingerprints(smiles, radius=2, n_bits=2048):
    gen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)
    out = []
    for smi in smiles:
        mol = Chem.MolFromSmiles(smi) if isinstance(smi, str) else None
        out.append(gen.GetFingerprint(mol) if mol is not None else None)
    return out


fp_train = morgan_fingerprints(train["SMILES"])
rows = []
for source, df in ext.items():
    smi = df["SMILES"].dropna()
    smi = smi.sample(min(1500, len(smi)), random_state=0)
    nn_similarity = common.nearest_neighbour_similarity(fp_train, morgan_fingerprints(smi))
    rows.append({"source": source,
                 "n": len(df),
                 "median NN sim to ExpansionRx": np.nanmedian(nn_similarity),
                 "% below noise floor":
                     100 * np.nanmean(nn_similarity < common.MORGAN2_NOISE_FLOOR)})
overlap = pd.DataFrame(rows).round(3)
overlap

- Which source sits closest to your chemistry?
- A source can sit right on top of your chemical space and still be useless.
  Which of these would you expect that of, and what would you check?

### Does it put a nearer neighbour next to the molecules I must predict?

A source can pass the test above and still be useless: if your own training
molecules are already closer to every validation compound, the external rows add
no coverage where you need it.

So this cell changes the question. For each **validation** molecule it measures
the nearest neighbour twice &mdash; once against the training fold alone, once
against the fold plus every external molecule pooled &mdash; and overlays the two
distributions. The gap between the histograms is the marginal value of all that
external data, for the predictions you are actually scored on.

In [ ]:
# For each validation molecule: how close is its nearest neighbour in the
# training set, before and after the external rows are stacked in?
# Reuses morgan_fingerprints from the cell above.

all_external = pd.concat([df["SMILES"] for df in ext.values()], ignore_index=True)

fp_val = morgan_fingerprints(val_df["SMILES"])
fp_our = morgan_fingerprints(train_df["SMILES"])
fp_ext = morgan_fingerprints(all_external)

nn_ours = common.nearest_neighbour_similarity(fp_val, fp_our)
nn_both = common.nearest_neighbour_similarity(fp_val, fp_our + fp_ext)

closer = nn_both > nn_ours + 1e-9
print(f"validation molecules                  : {len(nn_ours):,}")
print(f"median NN similarity, ours only       : {np.nanmedian(nn_ours):.3f}")
print(f"median NN similarity, ours + external : {np.nanmedian(nn_both):.3f}")
print(f"got a closer neighbour from the {len(all_external):,} external molecules: "
      f"{int(closer.sum())} ({100 * closer.mean():.1f}%)")

fig, ax = plt.subplots(figsize=(7, 3.6))
bins = np.linspace(0, 1, 41)
ax.hist(nn_ours, bins=bins, alpha=.55, label="nearest neighbour in ExpansionRx train")
ax.hist(nn_both, bins=bins, alpha=.55, label="...plus every external molecule")
ax.axvline(common.MORGAN2_NOISE_FLOOR, ls="--", c="k", lw=1)
ax.text(common.MORGAN2_NOISE_FLOOR, ax.get_ylim()[1] * .88, " noise floor", fontsize=8)
ax.set_xlabel("Tanimoto similarity to nearest training molecule")
ax.set_ylabel("validation molecules")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

- Compare the two histograms. How far did the external rows move the
  distribution?
- Read this against the table above. A source can score well there and add
  nothing here. What has to be true of your own training set for that to
  happen?
- If external rows cannot put a nearer neighbour next to a validation
  molecule, by what mechanism could they still improve the model? Is that
  mechanism plausible for a tree model on physicochemical descriptors?
- This pools every source into one picture. `ext[name]` would let you ask the
  question one source at a time &mdash; does the answer change?